<div style="background: linear-gradient(135deg, #172940 0%, #0854a0 60%, #0f828f 130%); color: #fff; padding: 40px 36px; border-radius: 14px; margin-bottom: 8px;">
  <p style="text-transform: uppercase; letter-spacing: 2.5px; font-size: 12px; font-weight: 700; opacity: .85; margin: 0 0 10px;">SAP Generative AI Hub · SAP Business Data Cloud · Hands-On Workshop · Module 3</p>
  <h1 style="font-size: 32px; margin: 0 0 8px; font-weight: 800;">Governed Business Context-Rich Answers from Multi-Source Data Products with SAP Business Data Cloud</h1>
  <p style="font-size: 18px; font-weight: 600; color: #ffd280; margin: 0 0 12px;">From Natural Language to a Business Context-Rich Multi-Source SAP & Non-SAP Data Foundation Platform with SAP Business Data Cloud</p>
  <p style="font-size: 15px; max-width: 760px; margin: 0; opacity: .93;">
    Ask historical and analytical questions about purchase orders, suppliers, and risk scores: powered by SAP Business Data Cloud data products and External Dataset <strong> · PO History · Supplier · Supplier Risk Category and Score</strong>
  </p>
</div>

### 🎯 Learning Objectives
After completing this module, participants will be able to:
- Understand how SAP Business Data Cloud data products serve as the governed data layer for AI-powered analytics, SAP-delivered data products (PurchaseOrder, Supplier) and custom products (Supplier Risk Score) are pre-modeled and ready to consume.
- Connect an AI agent to SAP Business Data Cloud through a Cloud Foundry proxy app using BTP Destination Service and OData services exposed from SAP Datasphere.
- Design governed business operations that map natural language questions to the right OData service: AM_PO_HISTORY for spend/delivery, V_SUPPLIER for master data, AM_SUPPLIER_RISK_SCORE for risk analytics.
- Author LLM tool schemas that reflect data product semantics:expose the right fields, guide the LLM to use pre-computed measures (e.g., CountPurchaseOrder) instead of counting raw rows, and set appropriate parameter constraints.
- Build a system prompt that teaches the LLM domain context:risk scoring logic (weighted dimensions), aggregation rules, and multi-service orchestration pattern.
- Orchestrate multi-round, multi-service LLM tool calls: the LLM autonomously chains calls across three services (e.g., get risk scores → resolve supplier names → combine into a single answer) without human intervention.
- Ask natural language questions over governed, multi-source data, from basic single-service queries to complex three-service analytical questions spanning PO history, supplier master, and risk scores.

### ⚙️ How It Works

You ask a natural language question. The LLM identifies the relevant context — supplier, material group,average price, total value, year, risk category, country etc. — from your question and queries the appropriate data product in SAP Business Data Cloud (OData Consumption API). No data is hardcoded — answers are assembled from purchase order history, supplier master data, and composite risk scores, all modeled and governed in SAP Datasphere.

| Operation | What it answers |
|---|---|
| `AM_PO_HISTORY` | Average Price, Total Value, Total Spend, Late Delivery for any material, material group, supplier |
| `V_SUPPLIER` |Supplier Name, City, Region, Country, Posting and Purchasing Blocks for any supplier  |
| `AM_SUPPLIER_RISK_SCORE` | Risk Category, Composite Risk Score, Political Stability, Logistics Reliability,Spend Concentration Score for any supplier or supplier and material group for a specific year period |

### 🏗️ Architecture: Module 3 SAP BDC Data Products Connectivity with Business context and semantics

```
┌─────────────────┐         HTTPS           ┌─────────────────────────────────────┐
│                 │ ──────────────────────► │  bdc_connectivity.py                │
│    Notebook     │                         │  (Cloud Foundry App on SAP BTP)     │
│                 │ ◄────────────────────── │                                     │
│                 │        JSON Response    └──────────┬──────────────────────────┘
└─────────────────┘                                    │
                                                       │  /odata
                                                       ▼
                                          ┌─────────────────────────────┐
                                          │  BTP Destination Service    │
                                          │  (Credentials + Routing)    │
                                          └──────────┬──────────────────┘
                                                     │
                                                     ▼
                                          ┌─────────────────────────────┐
                                          │    SAP Datasphere           │
                                          │    (OData V4 Service).      │
                                          └──────────┬──────────────────┘
                                                     │
                                                     ▼
┌────────────────────────────────────────────────────────────────────────────────┐
│                SAP Datasphere — Analytic Models & Views                        │
│                                                                                │
│  ┌──────────────────┐  ┌──────────────────┐  ┌─────────────────────────────┐   │
│  │ AM_PO_HISTORY    │  │ V_SUPPLIER       │  │ AM_SUPPLIER_RISK_SCORE      │   │
│  │ (PO Fact)        │  │ (Supplier Dim)   │  │ (Risk Fact)                 │   │
│  └────────┬─────────┘  └────────┬─────────┘  └──────────┬──────────────────┘   │
│           │                     │                       │                      │
│           │                     │                       │                      │ 
│           │                     │                       │                      │
│           ▼                     ▼                       ▼                      │
│  ┌─────────────────────────────────────────────────────────────────────────┐   │
│  │                SAP BDC Data Products (SAP Sources)                      │   │
│  │                                                                         │   │
│  │  ┌───────────────────┐ ┌───────────────────┐ ┌──────────────────────┐   │   │
│  │  │ PurchaseOrder     │ │PurchaseOrderItem  │ │PurchaseOrder         │   │   │
│  │  │                   │ │                   │ │ScheduleLine          │   │   │
│  │  └───────────────────┘ └───────────────────┘ └──────────────────────┘   │   │
│  │                                                                         │   │
│  │  ┌───────────────────┐                                                  │   │
│  │  │ Supplier          │                                                  │   │
│  │  └───────────────────┘                                                  │   │
│  └─────────────────────────────────────────────────────────────────────────┘   │
│                                                                                │
│  ┌─────────────────────────────────────────────────────────────────────────┐   │
│  │                External Data (Non-SAP Sources)                          │   │
│  │                                                                         │   │
│  │  ┌──────────────────────────────┐                                       │   │
│  │  │ EXT_COUNTRY_RISK_INDEX       │                                       │   │
│  │  │ (Country Risk Scores, CSV)   │                                       │   │
│  │  └──────────────────────────────┘                                       │   │
│  └─────────────────────────────────────────────────────────────────────────┘   │
└────────────────────────────────────────────────────────────────────────────────┘
```

### Data Lineage

```
SAP BDC Data Products (SAP)              External Data (Non-SAP)
├─ PurchaseOrder                         ├─ EXT_COUNTRY_RISK_INDEX
├─ PurchaseOrderItem                     │  (Country risk scores,
├─ PurchaseOrderScheduleLine             │   political stability,
├─ Supplier                              │   logistics reliability,
│                                        │   trade sanctions)
▼                                        ▼
┌────────────────────────────────────────────────────────────────┐
│                  SAP Datasphere Modeling                       │
│                                                                │
│  PO + POItem + POScheduleLine ──► HL_PO_HISTORY (Flattened)    │
│  Supplier ────────────────────► V_SUPPLIER (Dimension)         │
│                                                                │
│  HL_PO_HISTORY ──┐                                             │
│  V_SUPPLIER ─────┤──► V_SUPPLIER_RISK_SCORE (SQL View)         │
│  EXT_COUNTRY_ ───┘     ├─ Spend Concentration (35%)            │
│  RISK_INDEX            ├─ Single Source Dependency (30%)       │
│                        ├─ Country Risk (35%)                   │
│                        └─ Composite Risk Score (0-100)         │
│                                                                │
│  HL_PO_HISTORY ──────────────► AM_PO_HISTORY (Analytic Model)  │
│  V_SUPPLIER_RISK_SCORE ──────► AM_SUPPLIER_RISK_SCORE (AM)     │
└────────────────────────────────────────────────────────────────┘
```

### Request Flow

```
1. User asks a natural language question
2. Notebook sends HTTPS request to bdc_connectivity.py
3. App resolves /odata route → BTP Destination Service
4. Destination Service authenticates and routes to SAP Datasphere
5. Datasphere queries governed Analytic Models / Views
6. Models are built on SAP BDC data products + external data
7. Response flows back: Datasphere → Destination → App → Notebook → User
```


### Model Architecture
```
AM_PO_HISTORY          V_SUPPLIER          AM_SUPPLIER_RISK_SCORE
(Fact - PO detail)     (Dimension)         (Fact - Risk scores)
├─ Supplier ──────────→ Supplier ←─────────── Supplier
├─ Material            ├─ Country            ├─ Year
├─ MaterialGroup       ├─ Country_Name       ├─ CompositeRiskScore
├─ PO_Year             └─ ...                ├─ RiskCategory
├─ NetAmount                                 ├─ SpendConcentrationPct
└─ ...                                       ├─ SingleSourcedMaterialCount
                                             └─ ...

```

### 📋 Before You Start

Create or make sure a `.env` file in your project root with these values:

| Variable | Value |
|---|---|
| `AICORE_AUTH_URL` | Your AI Core auth URL |
| `AICORE_CLIENT_ID` | Your AI Core client ID |
| `AICORE_CLIENT_SECRET` | Your AI Core client secret |
| `GPT5_DEPLOYMENT_URL` | Your GPT-5 deployment URL |
| `DESTINATION` | BTP Destination name pointing to Datasphere |
| `BDC_PROXY_URL` | Cloud Foundry app route URL |

> Get these from your BTP cockpit → AI Core service key and Destination configuration.

> ✅ Checkpoint — Did Not Bring Your Own Tenant? The required variable and value details for the .env file can be found in [workshop_day_credentials.md](https://github.tools.sap/IA-C/2611_module3_bdc-main/blob/main/others/workshop_day_credentials.md) located in the `others` folder of this repository.


### Step 1: Install Python packages

Run the following package installations. **pip** is the package installer for Python. You can use pip to install packages from the Python Package Index and other indexes.

⚠️ **Note:** Jupyter/Python Notebook kernel restart required after package installation.

In [ ]:
%pip install -U python-dotenv pandas matplotlib requests
print("Done. Restart kernel then run from Step 2.")

⚠️ **Note:** After package installation, a kernel restart is required. See [Section 2, Step 4 — Configure Python Environment](https://github.tools.sap/IA-C/2611_module3_bdc-main/blob/main/module3_exercise_guide.md#section-2-configure-python-environment-and-open-jupyter-notebook) in the Exercise Guide for instructions.

### Step 2: Imports
Import all necessary modules and configure proper logging for the application.

**What we're importing:**
- `os`, `json`: For environment variables and JSON handling
- `logging`, `warnings`: For structured logging and warning suppression
- `typing`: For type hints to improve code clarity
- `dotenv`: To load environment variables from .env file

**Why logging matters:** Using the logging module instead of print statements provides better control over output levels, log formatting, and makes debugging easier in production environments.

In [92]:
import os
import json
import logging
import warnings
import re
from datetime import datetime
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
from IPython.display import Markdown, display

# Suppress deprecation warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Load environment variables
load_dotenv()

logger.info("Libraries imported and logging configured successfully!")
print("Libraries imported and logging configured successfully!")

2026-09-30 13:13:45,130 - __main__ - INFO - Libraries imported and logging configured successfully!


Libraries imported and logging configured successfully!


### Step 3: Configuration
Establish the connection chain from Notebook to SAP Business Data Cloud by loading AI Core credentials, Cloud Foundry proxy URL, and BTP Destination from a centralized `.env` file. Define the three OData service paths — AM_PO_HISTORY, V_SUPPLIER, and AM_SUPPLIER_RISK_SCORE — as reusable constants. This approach keeps all connection parameters in one place, enabling easy switching between environments without modifying code throughout the notebook.


In [ ]:
import os, json, time, datetime, requests
from typing import Optional
from IPython.display import display, Markdown
from dotenv import load_dotenv

load_dotenv()

# -- CF proxy app
BDC_PROXY_URL         = os.getenv("BDC_PROXY_URL")

# -- AI Core credentials from .env
AICORE_AUTH_URL       = os.getenv("AICORE_AUTH_URL")
AICORE_CLIENT_ID      = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET  = os.getenv("AICORE_CLIENT_SECRET")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP", "default")
GPT5_DEPLOYMENT_URL   = os.getenv("GPT5_DEPLOYMENT_URL")

# -- SAP Datasphere OData paths
DESTINATION     = os.getenv("DESTINATION")
REQUEST_TIMEOUT = 60

PO_HISTORY    = "/api/v1/datasphere/consumption/analytical/AI_DATA_FOUNDATION/AM_PO_HISTORY/AM_PO_HISTORY"
SUPPLIER      = "/api/v1/datasphere/consumption/relational/AI_DATA_FOUNDATION/V_SUPPLIER/V_SUPPLIER"
SUPPLIER_RISK = "/api/v1/datasphere/consumption/analytical/AI_DATA_FOUNDATION/AM_SUPPLIER_RISK_SCORE/AM_SUPPLIER_RISK_SCORE"

# -- Validate
missing = [k for k, v in {
    "AICORE_AUTH_URL":      AICORE_AUTH_URL,
    "AICORE_CLIENT_ID":     AICORE_CLIENT_ID,
    "AICORE_CLIENT_SECRET": AICORE_CLIENT_SECRET,
    "GPT5_DEPLOYMENT_URL":  GPT5_DEPLOYMENT_URL,
    "DESTINATION":          DESTINATION,
}.items() if not v]
if missing:
    raise EnvironmentError(f"Missing .env variables: {missing}")

print(f"CF App      : {BDC_PROXY_URL}")
print(f"Deployment  : {GPT5_DEPLOYMENT_URL.split('/')[-1]}")
print(f"Destination : {DESTINATION}")

CF App      : https://bdc-connection-palm-dugong-js.cfapps.eu10-004.hana.ondemand.com
Deployment  : dd903286e41187d5
Destination : BDC-AI-Data-Foundation


### Step 4 — AI Core LLM Client

Initialize a reusable client for SAP AI Core that handles OAuth token management, automatic token refresh, and chat completions with tool-calling support. This is the same client used in Module 2 — the LLM layer is environment-agnostic and works identically whether the backend is S/4HANA or SAP Business Data Cloud.


In [94]:
class GenAIHubClient:
    def __init__(self, auth_url, client_id, client_secret, resource_group, deployment_url):
        self._auth_url         = auth_url
        self._client_id        = client_id
        self._client_secret    = client_secret
        self._resource_group   = resource_group
        self._deployment_url   = deployment_url
        self._token            = None
        self._token_expires_at = 0.0

    def _get_token(self):
        if self._token and time.time() < self._token_expires_at - 60:
            return self._token
        r = requests.post(
            self._auth_url + "/oauth/token",
            data={"grant_type": "client_credentials"},
            auth=(self._client_id, self._client_secret),
            headers={"Content-Type": "application/x-www-form-urlencoded"},
            timeout=30,
        )
        r.raise_for_status()
        p = r.json()
        self._token = p["access_token"]
        self._token_expires_at = time.time() + int(p.get("expires_in", 600))
        return self._token

    def chat(self, messages, tools=None, tool_choice=None):
        body = {"messages": messages}
        if tools:
            body["tools"] = tools
            body["tool_choice"] = tool_choice or "auto"
        r = requests.post(
            self._deployment_url + "/v1/chat/completions",
            json=body,
            headers={
                "Authorization":     f"Bearer {self._get_token()}",
                "AI-Resource-Group": self._resource_group,
                "Content-Type":      "application/json",
            },
            timeout=60,
        )
        if not r.ok:
            raise Exception(f"LLM {r.status_code}: {r.text[:300]}")
        return r.json()


llm = GenAIHubClient(
    auth_url=AICORE_AUTH_URL, client_id=AICORE_CLIENT_ID,
    client_secret=AICORE_CLIENT_SECRET, resource_group=AICORE_RESOURCE_GROUP,
    deployment_url=GPT5_DEPLOYMENT_URL,
)
test = llm.chat([{"role": "user", "content": "Reply with exactly: AI Core ready."}])
print(f"LLM: {test['choices'][0]['message']['content'].strip()}")

LLM: AI Core ready.


### Step 5 — SAP BDC OData Helper

Define a single reusable function `bdc_get()` that routes all OData requests through the Cloud Foundry proxy app to SAP Datasphere. Unlike Module 2's `s4_get()` which required an SAP client number for S/4HANA, this helper only needs a BTP Destination name — Datasphere handles authentication and routing transparently. All three operations in Step 7 use this one function to reach their respective data products.


In [95]:
def bdc_get(path: str, params: dict = None) -> dict:
    """Call CF deployed app /odata endpoint — handles all BDC/Datasphere connectivity."""
    r = requests.get(
        f"{BDC_PROXY_URL}/odata",
        params={"dest": DESTINATION, "path": path, **(params or {})},
        timeout=REQUEST_TIMEOUT,
    )
    r.raise_for_status()
    return r.json()

print("bdc_get() defined.")

bdc_get() defined.


### Step 6 — Smoke Testing OData Calls

Validate end-to-end connectivity to all three SAP Business Data Cloud data products before building any logic on top. Each test fetches a single record from AM_PO_HISTORY, V_SUPPLIER, and AM_SUPPLIER_RISK_SCORE — confirming that the Cloud Foundry proxy, BTP Destination, and Datasphere OData services are all reachable and returning the expected field structure. If any test fails here, fix the connection before proceeding — everything downstream depends on these three calls working.


In [96]:
print("=== SVC1: AM_PO_HISTORY ===")
try:
    r = bdc_get(PO_HISTORY, {"$top": "1"})
    recs = r.get("value", [])
    print(f"PASSED — {len(recs)} record(s)")
    if recs: print(f"Fields: {list(recs[0].keys())}")
except Exception as e:
    print(f"FAILED: {e}")

print()
print("=== SVC2: V_SUPPLIER ===")
try:
    r = bdc_get(SUPPLIER, {"$top": "1"})
    recs = r.get("value", [])
    print(f"PASSED — {len(recs)} record(s)")
    if recs: print(f"Fields: {list(recs[0].keys())}")
except Exception as e:
    print(f"FAILED: {e}")

print()
print("=== SVC3: AM_SUPPLIER_RISK_SCORE ===")
try:
    r = bdc_get(SUPPLIER_RISK, {"$top": "1"})
    recs = r.get("value", [])
    print(f"PASSED — {len(recs)} record(s)")
    if recs: print(f"Fields: {list(recs[0].keys())}")
except Exception as e:
    print(f"FAILED: {e}")

=== SVC1: AM_PO_HISTORY ===
PASSED — 1 record(s)
Fields: ['CountPurchaseOrder', 'CountScheduleLines', 'IsCompletelyDelivered', 'IsLateDelivery', 'Late_Delivery_Pct', 'LocalCurrency', 'Material', 'MaterialGroup', 'NetAmount', 'NetPriceAmount', 'NetPriceQuantity', 'OrderQuantity', 'Plant', 'PurchaseOrder', 'PurchaseOrderDate', 'PurchaseOrderItem', 'PurchaseOrderQuantityUnit', 'PurchaseOrderScheduleLine', 'RoughGoodsReceiptQty', 'SchedLineStscDeliveryDate', 'ScheduleLineDeliveryDate', 'ScheduleLineOrderQuantity', 'Supplier', 'Unit_Price']

=== SVC2: V_SUPPLIER ===
PASSED — 1 record(s)
Fields: ['CityName', 'Country', 'PostingIsBlocked', 'PurchasingIsBlocked', 'Region', 'Supplier', 'SupplierName']

=== SVC3: AM_SUPPLIER_RISK_SCORE ===
PASSED — 1 record(s)
Fields: ['CompositeRiskScore', 'Country', 'Logistics_Reliability', 'Political_Stability', 'RiskCategory', 'SingleSourceScore', 'SingleSourcedMaterialCount', 'SpendConcentrationPct', 'SpendConcentrationScore', 'Supplier', 'SupplierGroupSpen

### Step 7 — Governed Business Operations

Define three purpose-built Python operations: one per data product(OData Consumption Service), that translate structured parameters into OData queries against SAP Business Data Cloud. `get_po_history()` serves spend, delivery, and pricing questions from AM_PO_HISTORY. `get_supplier_info()` provides supplier master data from V_SUPPLIER. `get_supplier_risk()` retrieves composite risk scores and sub-dimensions from AM_SUPPLIER_RISK_SCORE. Each operation applies a hard cap on row count, builds OData filters dynamically from the parameters it receives, and returns a standardized response with source attribution and retrieval timestamp. **The LLM never constructs OData queries directly, these operations act as the governed interface between natural language and live data**.


In [97]:
def _now() -> str:
    return datetime.datetime.utcnow().isoformat() + "Z"


# ==============================================================================
#  Operation 1: get_po_history
#  Service : AM_PO_HISTORY (Analytical)
#  Serves  : Q1, Q3, Q4, Q5
#
#  Fields  : PurchaseOrder, PurchaseOrderItem, PurchaseOrderScheduleLine,
#            PurchaseOrderDate, Supplier, Material, MaterialGroup, Plant,
#            NetAmount, NetPriceAmount, NetPriceQuantity, Unit_Price,
#            OrderQuantity, ScheduleLineOrderQuantity, RoughGoodsReceiptQty,
#            ScheduleLineDeliveryDate, SchedLineStscDeliveryDate,
#            IsCompletelyDelivered, IsLateDelivery, Late_Delivery_Pct,
#            CountPurchaseOrder, CountScheduleLines,
#            PurchaseOrderQuantityUnit, LocalCurrency
# ==============================================================================

def get_po_history(
    supplier:         Optional[str]   = None,
    year:             Optional[str]   = None,
    material:         Optional[str]   = None,
    material_group:   Optional[str]   = None,
    plant:            Optional[str]   = None,
    late_delivery_gt: Optional[float] = None,
    order_by:         Optional[str]   = None,
    top:              int             = 100,
    select:           Optional[str]   = None,
) -> dict:
    """Retrieve PO line data from AM_PO_HISTORY."""
    top = min(top, 5000)  # Hard cap

    filter_parts = []
    filters_log  = {}

    if supplier:
        filter_parts.append(f"Supplier eq '{supplier}'")
        filters_log["supplier"] = supplier
    if year:
        filter_parts.append(
            f"PurchaseOrderDate ge {year}-01-01 and PurchaseOrderDate le {year}-12-31"
        )
        filters_log["year"] = year
    if material:
        filter_parts.append(f"Material eq '{material}'")
        filters_log["material"] = material
    if material_group:
        filter_parts.append(f"MaterialGroup eq '{material_group}'")
        filters_log["material_group"] = material_group
    if plant:
        filter_parts.append(f"Plant eq '{plant}'")
        filters_log["plant"] = plant
    if late_delivery_gt is not None:
        filter_parts.append(f"Late_Delivery_Pct gt {late_delivery_gt}")
        filters_log["late_delivery_gt"] = late_delivery_gt

    params = {"$top": str(top)}
    if filter_parts:
        params["$filter"] = " and ".join(filter_parts)
    if order_by:
        params["$orderby"] = order_by
    if select:
        params["$select"] = select

    try:
        raw  = bdc_get(PO_HISTORY, params)
        rows = raw.get("value", [])
        return {
            "status": "ok", "retrieved_at": _now(),
            "source": "SAP Business Data Cloud - AM_PO_HISTORY",
            "filters_applied": filters_log,
            "row_count": len(rows), "data": rows,
        }
    except Exception as e:
        return {
            "status": "error", "retrieved_at": _now(),
            "source": "SAP Business Data Cloud - AM_PO_HISTORY",
            "filters_applied": filters_log, "error": str(e), "data": [],
        }


# ==============================================================================
#  Operation 2: get_supplier_info
#  Service : V_SUPPLIER (Relational)
#  Serves  : Q2, Q5 (enrichment), Q6 (enrichment), Q7 (enrichment)
#
#  Fields  : Supplier, SupplierName, Country, Region, CityName,
#            PostingIsBlocked, PurchasingIsBlocked
# ==============================================================================

def get_supplier_info(
    supplier: Optional[str] = None,
    country:  Optional[str] = None,
    top:      int           = 200,
    select:   Optional[str] = None,
) -> dict:
    """Retrieve supplier master records from V_SUPPLIER."""
    top = min(top, 1000)  # Hard cap

    filter_parts = []
    filters_log  = {}

    if supplier:
        filter_parts.append(f"Supplier eq '{supplier}'")
        filters_log["supplier"] = supplier
    if country:
        filter_parts.append(f"Country eq '{country}'")
        filters_log["country"] = country

    params = {"$top": str(top)}
    if filter_parts:
        params["$filter"] = " and ".join(filter_parts)
    if select:
        params["$select"] = select

    try:
        raw  = bdc_get(SUPPLIER, params)
        rows = raw.get("value", [])
        return {
            "status": "ok", "retrieved_at": _now(),
            "source": "SAP Business Data Cloud - V_SUPPLIER",
            "filters_applied": filters_log,
            "row_count": len(rows), "data": rows,
        }
    except Exception as e:
        return {
            "status": "error", "retrieved_at": _now(),
            "source": "SAP Business Data Cloud - V_SUPPLIER",
            "filters_applied": filters_log, "error": str(e), "data": [],
        }


# ==============================================================================
#  Operation 3: get_supplier_risk
#  Service : AM_SUPPLIER_RISK_SCORE (Analytical)
#  Serves  : Q6, Q7, Q8, Q9, Q10
#
#  Fields  : Supplier, Year, Country, CompositeRiskScore, RiskCategory,
#            SpendConcentrationScore, SpendConcentrationPct,
#            SingleSourceScore, SingleSourcedMaterialCount,
#            SupplierGroupSpend, TotalGroupSpend,
#            WorstMaterialGroup, WorstMaterialGroupName,
#            Trade_Sanctions_Flag, Political_Stability, Logistics_Reliability
# ==============================================================================

def get_supplier_risk(
    supplier:       Optional[str]  = None,
    year:           Optional[str]  = None,
    years:          Optional[str]  = None,
    country:        Optional[str]  = None,
    countries:      Optional[str]  = None,
    risk_category:  Optional[str]  = None,
    sanctions_only: Optional[bool] = None,
    order_by:       str            = "CompositeRiskScore desc",
    top:            int            = 50,
    select:         Optional[str]  = None,
) -> dict:
    """Retrieve supplier risk scores from AM_SUPPLIER_RISK_SCORE."""
    top = min(top, 1000)  # Hard cap

    filter_parts = []
    filters_log  = {}

    if supplier:
        filter_parts.append(f"Supplier eq '{supplier}'")
        filters_log["supplier"] = supplier
    if year and not years:
        filter_parts.append(f"Year eq '{year}'")
        filters_log["year"] = year
    if years:
        year_list = [y.strip() for y in years.split(",")]
        or_clause = "(" + " or ".join(f"Year eq '{y}'" for y in year_list) + ")"
        filter_parts.append(or_clause)
        filters_log["years"] = year_list
    if country and not countries:
        filter_parts.append(f"Country eq '{country}'")
        filters_log["country"] = country
    if countries:
        country_list = [c.strip() for c in countries.split(",")]
        or_clause = "(" + " or ".join(f"Country eq '{c}'" for c in country_list) + ")"
        filter_parts.append(or_clause)
        filters_log["countries"] = country_list
    if risk_category:
        filter_parts.append(f"RiskCategory eq '{risk_category.upper()}'")
        filters_log["risk_category"] = risk_category.upper()
    if sanctions_only:
        filter_parts.append("Trade_Sanctions_Flag eq 'Y'")
        filters_log["sanctions_only"] = True

    params = {"$top": str(top), "$orderby": order_by}
    if filter_parts:
        params["$filter"] = " and ".join(filter_parts)
    if select:
        params["$select"] = select

    try:
        raw  = bdc_get(SUPPLIER_RISK, params)
        rows = raw.get("value", [])
        return {
            "status": "ok", "retrieved_at": _now(),
            "source": "SAP Business Data Cloud - AM_SUPPLIER_RISK_SCORE",
            "filters_applied": filters_log,
            "row_count": len(rows), "data": rows,
        }
    except Exception as e:
        return {
            "status": "error", "retrieved_at": _now(),
            "source": "SAP Business Data Cloud - AM_SUPPLIER_RISK_SCORE",
            "filters_applied": filters_log, "error": str(e), "data": [],
        }


print("Three operations defined: get_po_history, get_supplier_info, get_supplier_risk")


Three operations defined: get_po_history, get_supplier_info, get_supplier_risk


### Step 8 — LLM Tool Schemas
Register the three operations as callable tools for the LLM using OpenAI-compatible function-calling format. Each schema declares the operation's name, a detailed description listing every available field from the OData metadata, and typed parameters with constraints. This is where the LLM learns what data exists and how to query it — for example, it learns to use the pre-computed `CountPurchaseOrder` field instead of counting raw rows, and to use comma-separated values for multi-country or multi-year filters. Well-authored schemas are the difference between the LLM making precise single-call queries and wasteful trial-and-error.


In [98]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_po_history",
            "description": (
                "Get purchase order history from SAP Business Data Cloud (AM_PO_HISTORY). "
                "Available fields: "
                "PurchaseOrder, PurchaseOrderItem, PurchaseOrderScheduleLine, "
                "PurchaseOrderDate (Date), Supplier, Material, MaterialGroup, Plant, "
                "NetAmount (line spend), NetPriceAmount, NetPriceQuantity, Unit_Price, "
                "OrderQuantity, ScheduleLineOrderQuantity, RoughGoodsReceiptQty, "
                "ScheduleLineDeliveryDate, SchedLineStscDeliveryDate, "
                "IsCompletelyDelivered (Boolean), IsLateDelivery (0/1), Late_Delivery_Pct (%), "
                "CountPurchaseOrder (pre-computed PO count per row), "
                "CountScheduleLines (pre-computed schedule line count), "
                "PurchaseOrderQuantityUnit, LocalCurrency. "
                "IMPORTANT: Use CountPurchaseOrder for PO counts - do NOT count raw rows. "
                "Use select to request only the fields you need."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "supplier":         {"type": "string", "description": "SAP supplier ID (e.g. '17300001'). Optional."},
                    "year":             {"type": "string", "description": "Calendar year YYYY (e.g. '2020'). Filters PurchaseOrderDate range."},
                    "material":         {"type": "string", "description": "SAP material number. Optional."},
                    "material_group":   {"type": "string", "description": "Material group code. Optional."},
                    "plant":            {"type": "string", "description": "SAP plant code. Optional."},
                    "late_delivery_gt": {"type": "number", "description": "Filter: only rows where Late_Delivery_Pct > this value (e.g. 1.0 for >1%)."},
                    "order_by":         {"type": "string", "description": "OData $orderby (e.g. 'NetAmount desc'). Optional."},
                    "top":              {"type": "integer", "description": "Max rows to return. Default 100. Max 5000."},
                    "select":           {"type": "string", "description": "Comma-separated field names to return. Optional."},
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_supplier_info",
            "description": (
                "Get supplier master data from SAP Business Data Cloud (V_SUPPLIER). "
                "Available fields: "
                "Supplier, SupplierName, Country, Region, CityName, "
                "PostingIsBlocked (Boolean), PurchasingIsBlocked (Boolean). "
                "Use for supplier counts by country, supplier names, locations, "
                "blocked status, or to enrich other results with supplier names."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "supplier": {"type": "string", "description": "SAP supplier ID. Optional."},
                    "country":  {"type": "string", "description": "Country code (e.g. 'US'). Optional."},
                    "top":      {"type": "integer", "description": "Max rows. Default 200. Max 1000."},
                    "select":   {"type": "string", "description": "Comma-separated field names. Optional."},
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_supplier_risk",
            "description": (
                "Get supplier risk scores from SAP Business Data Cloud (AM_SUPPLIER_RISK_SCORE). "
                "Available fields: "
                "Supplier, Year (YYYY string), Country, "
                "CompositeRiskScore (0-100 weighted score), "
                "RiskCategory (HIGH/MEDIUM/LOW), "
                "SpendConcentrationScore (0-100, weight 35%), "
                "SpendConcentrationPct (supplier spend / total group spend * 100), "
                "SingleSourceScore (0-100, weight 30%), "
                "SingleSourcedMaterialCount (materials with only this supplier), "
                "SupplierGroupSpend, TotalGroupSpend, "
                "WorstMaterialGroup, WorstMaterialGroupName, "
                "Trade_Sanctions_Flag (Y/N - if Y then CompositeRiskScore=99), "
                "Political_Stability (0-100), Logistics_Reliability (0-100). "
                "Each row = one supplier for one year with all risk dimensions. "
                "Scoring: Composite = SpendConcentration(35%) + SingleSource(30%) + CountryRisk(35%)."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "supplier":       {"type": "string", "description": "SAP supplier ID. Optional."},
                    "year":           {"type": "string", "description": "Single year YYYY (e.g. '2019'). Optional."},
                    "years":          {"type": "string", "description": "Comma-separated years for trends (e.g. '2018,2019,2020'). Optional."},
                    "country":        {"type": "string", "description": "Single country code (e.g. 'US'). Optional."},
                    "countries":      {"type": "string", "description": "Comma-separated country codes (e.g. 'US,IN'). Optional."},
                    "risk_category":  {"type": "string", "description": "'HIGH', 'MEDIUM', or 'LOW'. Optional."},
                    "sanctions_only": {"type": "boolean", "description": "True = only suppliers with Trade_Sanctions_Flag='Y'. Optional."},
                    "order_by":       {"type": "string", "description": "OData $orderby. Default: 'CompositeRiskScore desc'."},
                    "top":            {"type": "integer", "description": "Max rows. Default 50. Max 1000."},
                    "select":         {"type": "string", "description": "Comma-separated field names. Optional."},
                },
                "required": []
            }
        }
    },
]

print("Tool schemas defined: get_po_history, get_supplier_info, get_supplier_risk")


Tool schemas defined: get_po_history, get_supplier_info, get_supplier_risk


### Step 9: System Prompt

In [99]:
today_str = datetime.date.today().isoformat()

SYSTEM_PROMPT = f"""You are a procurement analytics assistant connected to SAP Business Data Cloud.
Today is {today_str}. Never fabricate data — always retrieve from the tools.

You have THREE read-only operations:

  1. get_po_history    — Purchase order line data: spend, quantities, materials,
                         delivery dates, late delivery %, pricing, PO counts.
                         Source: AM_PO_HISTORY (SAP BDC Data Product).
                         Key fields: Material, Supplier, NetAmount, Unit_Price,
                         CountPurchaseOrder, CountScheduleLines, Late_Delivery_Pct,
                         PurchaseOrderDate, MaterialGroup, Plant, LocalCurrency.

  2. get_supplier_info — Supplier master data: name, country, city, region,
                         blocked status.
                         Source: V_SUPPLIER (SAP BDC Data Product).
                         Key fields: Supplier, SupplierName, Country, Region,
                         CityName, PostingIsBlocked, PurchasingIsBlocked.

  3. get_supplier_risk — Supplier risk scores: composite score (0-100),
                         risk category, spend concentration, single source
                         dependency, country risk, sanctions flag.
                         Source: AM_SUPPLIER_RISK_SCORE (SAP BDC Data Product).
                         Key fields: Supplier, Year, Country, CompositeRiskScore,
                         RiskCategory, SpendConcentrationScore, SpendConcentrationPct,
                         SingleSourceScore, SingleSourcedMaterialCount,
                         SupplierGroupSpend, TotalGroupSpend, WorstMaterialGroup,
                         Trade_Sanctions_Flag, Political_Stability, Logistics_Reliability.

PARAMETER RULES:
- Extract parameters from the user's question. Never guess identifiers.
- Use 'select' to request ONLY the fields you need — keeps payloads small.
- For get_po_history: pass 'year' as YYYY string. Use 'order_by' and 'top' for ranking.
- For get_supplier_risk: use 'year' for single year, 'years' (comma-separated) for trends.
  Use 'countries' (comma-separated) for multi-country filters.
- For multi-step questions: call multiple operations in parallel when possible.
- Never request more than 5000 rows. The orchestrator pre-aggregates large results for you.

AGGREGATION RULES:
- These operations return row-level data. YOU must aggregate in your answer.
- "how many POs" → sum CountPurchaseOrder values (pre-computed field). Do NOT count raw rows.
- "total spend" → sum NetAmount values.
- "average price" → mean of Unit_Price or NetPriceAmount.
- "top N materials" → group by Material, sum NetAmount, rank descending.
- "suppliers per country" → group by Country, count distinct Supplier.
- If the orchestrator provides pre_aggregated data, use that directly.
- Always show numbers in a markdown table when comparing multiple items.

RISK SCORE INTERPRETATION:
- CompositeRiskScore = SpendConcentration(35%) + SingleSource(30%) + CountryRisk(35%).
- "what drives the risk" → compare SpendConcentrationScore, SingleSourceScore,
  and Political_Stability/Logistics_Reliability. Highest sub-score = dominant driver.
- Trade_Sanctions_Flag = 'Y' → CompositeRiskScore overridden to 99.

MULTI-STEP RULES:
- When the user asks for supplier names and you only have supplier IDs from another operation,
  ALWAYS call get_supplier_info automatically to enrich the results. Never ask the user for
  confirmation to make a second call — just do it.
- When multiple operations are needed, call them all. Do not stop halfway and ask permission.

FORMATTING RULES:
- When presenting tables, ALWAYS use proper markdown format with header separators:
  | Column1 | Column2 |
  |---------|---------|
  | value1  | value2  |
- Format large numbers with commas (e.g. 1,234,567.89).
- Format percentages with 1 decimal place (e.g. 45.3%).
- Always include source system and retrieval timestamp at the end.
- Never fabricate identifiers (supplier ID, material number, plant).
- Write operations: decline clearly.
- Out-of-scope requests: short clear refusal only.
"""

print("System prompt ready!")


System prompt ready!


### Step 10 — Query Orchestrator (Ask Engine)
The central `ask()` function that ties everything together — takes a natural language question in, returns a sourced analytical answer out. It manages a multi-round conversation loop: the LLM decides which operations to call, the orchestrator executes them against SAP Business Data Cloud, and feeds the results back for the LLM to either make additional calls or compose the final answer. A built-in payload manager `_summarize_for_llm()` pre-aggregates large result sets — grouping by country, summing spend by material, computing PO counts — so the LLM receives compact summaries instead of thousands of raw rows. This keeps responses fast, token-efficient, and within timeout limits.


In [100]:
# ==============================================================================
#  STEP 10 — Orchestrator (multi-round tool calling + smart payload management)
# ==============================================================================

OPERATIONS = {
    "get_po_history":    get_po_history,
    "get_supplier_info": get_supplier_info,
    "get_supplier_risk": get_supplier_risk,
}

MAX_ROWS_FOR_LLM = 100
MAX_ROUNDS       = 3


def _summarize_for_llm(result: dict) -> dict:
    """
    If result exceeds MAX_ROWS_FOR_LLM, pre-aggregate common patterns
    so the LLM gets a compact payload instead of thousands of raw rows.
    """
    data = result.get("data", [])
    if not isinstance(data, list) or len(data) <= MAX_ROWS_FOR_LLM:
        return result

    result = dict(result)
    total_rows = len(data)

    try:
        df = pd.DataFrame(data)
        summary = {}

        # -- Supplier count by country (V_SUPPLIER) ---------------------------
        if "Country" in df.columns and "Supplier" in df.columns:
            summary["suppliers_per_country"] = (
                df.groupby("Country")["Supplier"].nunique()
                .sort_values(ascending=False).to_dict()
            )

        # -- Spend and PO count by material (AM_PO_HISTORY) -------------------
        if "Material" in df.columns and "NetAmount" in df.columns:
            df["NetAmount"] = pd.to_numeric(df["NetAmount"], errors="coerce")
            mat_group = df.groupby("Material")

            summary["spend_by_material"] = (
                mat_group["NetAmount"].sum()
                .sort_values(ascending=False).head(20).to_dict()
            )
            if "CountPurchaseOrder" in df.columns:
                df["CountPurchaseOrder"] = pd.to_numeric(df["CountPurchaseOrder"], errors="coerce")
                summary["po_count_by_material"] = (
                    mat_group["CountPurchaseOrder"].sum().to_dict()
                )

        # -- Spend and PO count by supplier (AM_PO_HISTORY) -------------------
        if "Supplier" in df.columns and "NetAmount" in df.columns:
            df["NetAmount"] = pd.to_numeric(df["NetAmount"], errors="coerce")
            sup_group = df.groupby("Supplier")

            summary["spend_by_supplier"] = (
                sup_group["NetAmount"].sum()
                .sort_values(ascending=False).head(20).to_dict()
            )
            if "CountPurchaseOrder" in df.columns:
                df["CountPurchaseOrder"] = pd.to_numeric(df["CountPurchaseOrder"], errors="coerce")
                summary["po_count_by_supplier"] = (
                    sup_group["CountPurchaseOrder"].sum().to_dict()
                )

        # -- Late delivery by supplier (AM_PO_HISTORY) ------------------------
        if "Supplier" in df.columns and "Late_Delivery_Pct" in df.columns:
            df["Late_Delivery_Pct"] = pd.to_numeric(df["Late_Delivery_Pct"], errors="coerce")
            summary["late_delivery_by_supplier"] = (
                df.groupby("Supplier")["Late_Delivery_Pct"].mean()
                .sort_values(ascending=False).head(20).to_dict()
            )

        # -- Price by supplier for a material (AM_PO_HISTORY) -----------------
        if "Supplier" in df.columns and "Unit_Price" in df.columns and "Material" in df.columns:
            df["Unit_Price"] = pd.to_numeric(df["Unit_Price"], errors="coerce")
            summary["avg_price_by_supplier"] = (
                df.groupby("Supplier")["Unit_Price"].mean()
                .sort_values(ascending=True).to_dict()
            )

        if summary:
            result["pre_aggregated"] = summary
            result["data"] = []
            result["note"] = f"Pre-aggregated from {total_rows} rows. Use pre_aggregated for your answer."
        else:
            result["data"] = data[:MAX_ROWS_FOR_LLM]
            result["note"] = f"Truncated: showing {MAX_ROWS_FOR_LLM} of {total_rows} rows."

    except Exception:
        result["data"] = data[:MAX_ROWS_FOR_LLM]
        result["note"] = f"Truncated: showing {MAX_ROWS_FOR_LLM} of {total_rows} rows."

    result["row_count"] = total_rows
    return result


def ask(question: str, verbose: bool = True) -> str:
    """Natural language -> live SAP Business Data Cloud answer."""
    if verbose:
        print(f"\n{'='*70}")
        print(f"Q: {question}")
        print(f"{'='*70}")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]

    for round_num in range(MAX_ROUNDS):

        # -- LLM call: decide tool(s) or give final answer --------------------
        try:
            r = llm.chat(messages, tools=TOOLS, tool_choice="auto")
        except Exception as e:
            return f"LLM error: {e}"

        msg        = r["choices"][0]["message"]
        tool_calls = msg.get("tool_calls")

        # -- No tool call → final answer --------------------------------------
        if not tool_calls:
            answer = msg.get("content", "")
            if verbose:
                print()
                display(Markdown(answer))
            return answer

        # -- Execute all tool calls for this round ----------------------------
        if verbose and round_num > 0:
            print(f"\n--- Round {round_num + 1} ---")

        tool_results = {}
        for tc in tool_calls:
            fn_name = tc["function"]["name"]
            params  = json.loads(tc["function"]["arguments"])

            if verbose:
                print(f"Operation  : {fn_name}")
                print(f"Parameters : {json.dumps(params, indent=2)}")

            fn = OPERATIONS.get(fn_name)
            if not fn:
                tool_results[tc["id"]] = {"error": f"Unknown operation: {fn_name}"}
                continue

            try:
                result = fn(**params)
                tool_results[tc["id"]] = result
                if verbose:
                    print(f"BDC Result : {result.get('row_count', 0)} row(s) from {result.get('source', 'unknown')}")
            except Exception as e:
                tool_results[tc["id"]] = {"error": str(e)}

        # -- Append assistant msg + tool results to conversation history -------
        assistant_msg = {
            "role":       "assistant",
            "content":    msg.get("content") or "",
            "tool_calls": tool_calls,
        }

        tool_messages = [
            {
                "role":         "tool",
                "tool_call_id": tc["id"],
                "content":      json.dumps(_summarize_for_llm(tool_results.get(tc["id"], {}))),
            }
            for tc in tool_calls
        ]

        messages += [assistant_msg] + tool_messages
        # Loop continues — LLM will see results and decide: call more tools or answer

    # -- Exhausted all rounds — force a final answer without tools -------------
    try:
        r = llm.chat(messages)
        answer = r["choices"][0]["message"]["content"]
    except Exception as e:
        return f"LLM verbalization error: {e}"

    if verbose:
        print()
        display(Markdown(answer))
    return answer


print("Orchestrator ready!")


Orchestrator ready!


### Step 11 — Testing: With Business Context of SAP Business Data Cloud

Put it all together — ask natural language questions against live SAP Business Data Cloud data products and watch the engine work. Ten questions progress from basic single-service lookups to complex multi-service analytical chains, demonstrating how the LLM autonomously selects the right operations, extracts parameters, chains results across services, and composes sourced answers. Replace supplier IDs, materials, and years with values from your system to explore your own data.

Replace the supplier ID, material, year etc below with values from your system.

#### Questions

In [90]:
# Q1 (Basic) — Single: AM_PO_HISTORY
ask("How many purchase orders did we place with Supplier USSU-VSF85 in 2026 — and what was the total value?")


Q: How many purchase orders did we place with Supplier USSU-VSF85 in 2026 — and what was the total value?
Operation  : get_po_history
Parameters : {
  "supplier": "USSU-VSF85",
  "year": "2026",
  "top": 5000,
  "select": "NetAmount,CountPurchaseOrder,LocalCurrency"
}
BDC Result : 1 row(s) from SAP Business Data Cloud - AM_PO_HISTORY



Here’s what we placed with Supplier USSU-VSF85 in calendar year 2026:
- Purchase orders: 476
- Total PO value: USD 119,000

Source: SAP Business Data Cloud — AM_PO_HISTORY (retrieved 2026-09-29T21:38:45Z)

'Here’s what we placed with Supplier USSU-VSF85 in calendar year 2026:\n- Purchase orders: 476\n- Total PO value: USD 119,000\n\nSource: SAP Business Data Cloud — AM_PO_HISTORY (retrieved 2026-09-29T21:38:45Z)'

In [57]:
# Q2 (Basic) — Single: V_SUPPLIER
ask("How many suppliers do we have in each country? Show the result as a table of country code and supplier count.")


Q: How many suppliers do we have in each country? Show the result as a table of country code and supplier count.
Operation  : get_supplier_info
Parameters : {
  "top": 1000,
  "select": "Supplier,Country"
}
BDC Result : 902 row(s) from SAP Business Data Cloud - V_SUPPLIER



Here’s the supplier count by country:

| Country | Supplier Count |
|---------|-----------------|
| US      | 668             |
| DE      | 98              |
| CN      | 23              |
| FR      | 20              |
| GB      | 18              |
| AU      | 13              |
| EG      | 11              |
| JP      | 11              |
| HU      | 9               |
| ID      | 9               |
| IN      | 8               |
| IT      | 4               |
| CZ      | 3               |
| PL      | 2               |
| CH      | 1               |
| BR      | 1               |
| HK      | 1               |
| SG      | 1               |
| AR      | 1               |

Source: SAP Business Data Cloud - V_SUPPLIER | Retrieved: 2026-09-29T20:01:57.937254Z

'Here’s the supplier count by country:\n\n| Country | Supplier Count |\n|---------|-----------------|\n| US      | 668             |\n| DE      | 98              |\n| CN      | 23              |\n| FR      | 20              |\n| GB      | 18              |\n| AU      | 13              |\n| EG      | 11              |\n| JP      | 11              |\n| HU      | 9               |\n| ID      | 9               |\n| IN      | 8               |\n| IT      | 4               |\n| CZ      | 3               |\n| PL      | 2               |\n| CH      | 1               |\n| BR      | 1               |\n| HK      | 1               |\n| SG      | 1               |\n| AR      | 1               |\n\nSource: SAP Business Data Cloud - V_SUPPLIER | Retrieved: 2026-09-29T20:01:57.937254Z'

In [59]:
# Q3 (Basic) — Single: AM_PO_HISTORY
ask("What are our top 3 materials by total spend in 2019 and how many POs were created for those top 3 materials in 2019?")


Q: What are our top 3 materials by total spend in 2019 and how many POs were created for those top 3 materials in 2019?
Operation  : get_po_history
Parameters : {
  "year": "2019",
  "order_by": "NetAmount desc",
  "top": 3,
  "select": "Material,NetAmount,CountPurchaseOrder"
}
BDC Result : 3 row(s) from SAP Business Data Cloud - AM_PO_HISTORY



| Rank | Material        | Total Spend | POs (Count) |
|------|------------------|-------------|-------------|
| 1    | MZ-RM-S100-01    | 51,901,369,500 | 12          |
| 2    | MZ-RM-R300-01    | 22,176,423.92  | 341         |
| 3    | MZ-RM-C990-01    | 20,419,898.05  | 346         |

Notes:
- Total spend is the sum of NetAmount for 2019.
- PO count is the pre-computed CountPurchaseOrder for 2019 (not the number of lines).

Source: SAP Business Data Cloud - AM_PO_HISTORY (retrieved 2026-09-29T20:04:13Z)

'| Rank | Material        | Total Spend | POs (Count) |\n|------|------------------|-------------|-------------|\n| 1    | MZ-RM-S100-01    | 51,901,369,500 | 12          |\n| 2    | MZ-RM-R300-01    | 22,176,423.92  | 341         |\n| 3    | MZ-RM-C990-01    | 20,419,898.05  | 346         |\n\nNotes:\n- Total spend is the sum of NetAmount for 2019.\n- PO count is the pre-computed CountPurchaseOrder for 2019 (not the number of lines).\n\nSource: SAP Business Data Cloud - AM_PO_HISTORY (retrieved 2026-09-29T20:04:13Z)'

In [60]:
# Q4 (Intermediate) — Single: AM_PO_HISTORY
ask("Which suppliers have delivered late more than 1% of the time in the year 2017?")


Q: Which suppliers have delivered late more than 1% of the time in the year 2017?
Operation  : get_po_history
Parameters : {
  "year": "2017",
  "select": "Supplier,Late_Delivery_Pct,CountScheduleLines",
  "top": 5000
}
BDC Result : 102 row(s) from SAP Business Data Cloud - AM_PO_HISTORY



Suppliers with late delivery rate >1% in 2017:

| Supplier     | Late delivery % |
|--------------|-----------------|
| LSUPP-0001   | 2.1%            |

All other suppliers in 2017 were at or below 1.0%.

Source: SAP Business Data Cloud - AM_PO_HISTORY (retrieved 2026-09-29T20:05:09.964797Z)

'Suppliers with late delivery rate >1% in 2017:\n\n| Supplier     | Late delivery % |\n|--------------|-----------------|\n| LSUPP-0001   | 2.1%            |\n\nAll other suppliers in 2017 were at or below 1.0%.\n\nSource: SAP Business Data Cloud - AM_PO_HISTORY (retrieved 2026-09-29T20:05:09.964797Z)'

In [63]:
# Q5 (Intermediate) — Two: AM_PO_HISTORY → V_SUPPLIER
ask("What is the average price we pay for material TG0011 from 2015 to 2017 across all suppliers and which supplier offers the lowest price and highest price?")


Q: What is the average price we pay for material TG0011 from 2015 to 2017 across all suppliers and which supplier offers the lowest price and highest price?
Operation  : get_po_history
Parameters : {
  "material": "TG0011",
  "year": "2015",
  "top": 5000,
  "select": "Supplier,Unit_Price,LocalCurrency"
}
BDC Result : 2 row(s) from SAP Business Data Cloud - AM_PO_HISTORY
Operation  : get_po_history
Parameters : {
  "material": "TG0011",
  "year": "2016",
  "top": 5000,
  "select": "Supplier,Unit_Price,LocalCurrency"
}
BDC Result : 3 row(s) from SAP Business Data Cloud - AM_PO_HISTORY
Operation  : get_po_history
Parameters : {
  "material": "TG0011",
  "year": "2017",
  "top": 5000,
  "select": "Supplier,Unit_Price,LocalCurrency"
}
BDC Result : 1 row(s) from SAP Business Data Cloud - AM_PO_HISTORY



I pulled purchase order history for material TG0011 for 2015–2017 and computed the simple average of Unit_Price across all suppliers. Because the data is in multiple local currencies, I report averages and price extrema by currency (no FX conversion applied).

Average unit price (2015–2017)
| Currency | Average Unit Price |
|----------|--------------------|
| USD      | 32.08              |
| EUR      | 32.50              |

Suppliers with lowest and highest observed unit price (2015–2017)
| Currency | Lowest Price | Supplier (Lowest) | Year | Highest Price | Supplier (Highest) | Year |
|----------|--------------|-------------------|------|---------------|--------------------|------|
| USD      | 30.00        | 0017300001        | 2015 | 35.33         | 0017300002         | 2015 |
| USD      | 30.00        | 0017300001        | 2016 | 33.00         | 0017300002         | 2016 |
| EUR      | 30.00        | 0010300001        | 2016 | 35.00         | 0010300002         | 2017 |

Notes:
- The averages are simple means of Unit_Price records returned for TG0011 in each currency across 2015–2017 (not weighted by quantities).
- If you need a single overall average across currencies, please specify a target currency and FX convention (e.g., monthly average rate, booking-date rate), and I can recompute. I can also provide quantity-weighted averages if preferred.

Source: SAP Business Data Cloud — AM_PO_HISTORY (retrieved 2026-09-29T20:12:40Z, 20:12:42Z, 20:12:45Z)

'I pulled purchase order history for material TG0011 for 2015–2017 and computed the simple average of Unit_Price across all suppliers. Because the data is in multiple local currencies, I report averages and price extrema by currency (no FX conversion applied).\n\nAverage unit price (2015–2017)\n| Currency | Average Unit Price |\n|----------|--------------------|\n| USD      | 32.08              |\n| EUR      | 32.50              |\n\nSuppliers with lowest and highest observed unit price (2015–2017)\n| Currency | Lowest Price | Supplier (Lowest) | Year | Highest Price | Supplier (Highest) | Year |\n|----------|--------------|-------------------|------|---------------|--------------------|------|\n| USD      | 30.00        | 0017300001        | 2015 | 35.33         | 0017300002         | 2015 |\n| USD      | 30.00        | 0017300001        | 2016 | 33.00         | 0017300002         | 2016 |\n| EUR      | 30.00        | 0010300001        | 2016 | 35.00         | 0010300002         | 201

In [61]:
# Q6 (Intermediate) — Two: AM_SUPPLIER_RISK_SCORE → V_SUPPLIER
ask("Are any of our suppliers located in sanctioned countries?")


Q: Are any of our suppliers located in sanctioned countries?
Operation  : get_supplier_risk
Parameters : {
  "year": "2026",
  "sanctions_only": true,
  "top": 1000,
  "select": "Supplier,Country,Trade_Sanctions_Flag,RiskCategory,CompositeRiskScore",
  "order_by": "CompositeRiskScore desc"
}
BDC Result : 0 row(s) from SAP Business Data Cloud - AM_SUPPLIER_RISK_SCORE



Short answer: No. Based on the 2026 supplier risk dataset, none of your suppliers are flagged as being in sanctioned countries (Trade_Sanctions_Flag = Y returned 0 results).

If you’d like, I can also check prior years or monitor for changes going forward.

Source: SAP Business Data Cloud — AM_SUPPLIER_RISK_SCORE (retrieved 2026-09-29T20:07:25Z)

'Short answer: No. Based on the 2026 supplier risk dataset, none of your suppliers are flagged as being in sanctioned countries (Trade_Sanctions_Flag = Y returned 0 results).\n\nIf you’d like, I can also check prior years or monitor for changes going forward.\n\nSource: SAP Business Data Cloud — AM_SUPPLIER_RISK_SCORE (retrieved 2026-09-29T20:07:25Z)'

In [68]:
# Q7 (Intermediate) — Two: AM_SUPPLIER_RISK_SCORE → V_SUPPLIER
ask("Which suppliers are rated HIGH risk in USA and India in 2019? - Please provide supplier name aswell")


Q: Which suppliers are rated HIGH risk in USA and India in 2019? - Please provide supplier name aswell
Operation  : get_supplier_risk
Parameters : {
  "year": "2019",
  "countries": "US,IN",
  "risk_category": "HIGH",
  "order_by": "CompositeRiskScore desc",
  "top": 1000,
  "select": "Supplier,Country,CompositeRiskScore,RiskCategory"
}
BDC Result : 7 row(s) from SAP Business Data Cloud - AM_SUPPLIER_RISK_SCORE

--- Round 2 ---
Operation  : get_supplier_info
Parameters : {
  "supplier": "ABD",
  "select": "Supplier,SupplierName,Country"
}
BDC Result : 1 row(s) from SAP Business Data Cloud - V_SUPPLIER
Operation  : get_supplier_info
Parameters : {
  "supplier": "RSU300100",
  "select": "Supplier,SupplierName,Country"
}
BDC Result : 1 row(s) from SAP Business Data Cloud - V_SUPPLIER
Operation  : get_supplier_info
Parameters : {
  "supplier": "RSU300102",
  "select": "Supplier,SupplierName,Country"
}
BDC Result : 1 row(s) from SAP Business Data Cloud - V_SUPPLIER
Operation  : get_supplie

Here are the suppliers rated HIGH risk in the USA and India for 2019, with names included:

| Supplier ID  | Supplier Name               | Country | 2019 Risk Category | Composite Risk Score |
|--------------|-----------------------------|---------|--------------------|----------------------|
| ABD          | ABD Bearing                 | IN      | HIGH               | 67                   |
| RSU300100    | Northeastern Vendor         | US      | HIGH               | 65                   |
| RSU300102    | Midwestern Vendor           | US      | HIGH               | 65                   |
| RSU300104    | Southern Vendor             | US      | HIGH               | 65                   |
| USSU-VSF01   | EV Parts Inc.               | US      | HIGH               | 65                   |
| USSU-VSF02   | WaveCrest Labs              | US      | HIGH               | 65                   |
| V620         | Michigan Brake Components   | US      | HIGH               | 65                   |

Source: SAP Business Data Cloud — AM_SUPPLIER_RISK_SCORE (retrieved 2026-09-29T20:19:06Z) and V_SUPPLIER (retrieved 2026-09-29T20:19:21Z)

'Here are the suppliers rated HIGH risk in the USA and India for 2019, with names included:\n\n| Supplier ID  | Supplier Name               | Country | 2019 Risk Category | Composite Risk Score |\n|--------------|-----------------------------|---------|--------------------|----------------------|\n| ABD          | ABD Bearing                 | IN      | HIGH               | 67                   |\n| RSU300100    | Northeastern Vendor         | US      | HIGH               | 65                   |\n| RSU300102    | Midwestern Vendor           | US      | HIGH               | 65                   |\n| RSU300104    | Southern Vendor             | US      | HIGH               | 65                   |\n| USSU-VSF01   | EV Parts Inc.               | US      | HIGH               | 65                   |\n| USSU-VSF02   | WaveCrest Labs              | US      | HIGH               | 65                   |\n| V620         | Michigan Brake Components   | US      | HIGH               | 65         

In [70]:
# Q8 (Complex) — Two: AM_SUPPLIER_RISK_SCORE → V_SUPPLIER
ask("What is the driving score for the HIGH risk supplier ABD from India in 2019?")


Q: What is the driving score for the HIGH risk supplier ABD from India in 2019?
Operation  : get_supplier_info
Parameters : {
  "country": "IN",
  "top": 1000,
  "select": "Supplier,SupplierName,Country"
}
BDC Result : 8 row(s) from SAP Business Data Cloud - V_SUPPLIER

--- Round 2 ---
Operation  : get_supplier_risk
Parameters : {
  "supplier": "ABD",
  "year": "2019",
  "country": "IN",
  "top": 1,
  "select": "Supplier,Year,Country,CompositeRiskScore,RiskCategory,SpendConcentrationScore,SpendConcentrationPct,SingleSourceScore,SingleSourcedMaterialCount,Political_Stability,Logistics_Reliability,Trade_Sanctions_Flag,WorstMaterialGroup"
}
BDC Result : 1 row(s) from SAP Business Data Cloud - AM_SUPPLIER_RISK_SCORE



The dominant driver is Spend Concentration: score 90 for supplier ABD (India) in 2019, which makes it the driving score behind the HIGH risk rating. This is supported by a 100.0% spend concentration with ABD.

Key risk sub-metrics
| Metric                   | Value |
|--------------------------|-------|
| Spend Concentration      | 90    |
| Single Source            | 60    |
| Political Stability      | 56    |
| Logistics Reliability    | 58    |
| Composite Risk Score     | 67    |
| Trade Sanctions Flag     | N     |
| Spend Concentration Pct  | 100.0%|

Sources:
- SAP Business Data Cloud – AM_SUPPLIER_RISK_SCORE (retrieved 2026-09-29T20:21:28Z)
- SAP Business Data Cloud – V_SUPPLIER (retrieved 2026-09-29T20:21:12Z)

'The dominant driver is Spend Concentration: score 90 for supplier ABD (India) in 2019, which makes it the driving score behind the HIGH risk rating. This is supported by a 100.0% spend concentration with ABD.\n\nKey risk sub-metrics\n| Metric                   | Value |\n|--------------------------|-------|\n| Spend Concentration      | 90    |\n| Single Source            | 60    |\n| Political Stability      | 56    |\n| Logistics Reliability    | 58    |\n| Composite Risk Score     | 67    |\n| Trade Sanctions Flag     | N     |\n| Spend Concentration Pct  | 100.0%|\n\nSources:\n- SAP Business Data Cloud – AM_SUPPLIER_RISK_SCORE (retrieved 2026-09-29T20:21:28Z)\n- SAP Business Data Cloud – V_SUPPLIER (retrieved 2026-09-29T20:21:12Z)'

In [78]:
# Q9 (Complex) — Three: RISK → PO_HISTORY → V_SUPPLIER
ask("Who are our top 3 highest-risk suppliers in 2020? For each, show their supplier name, composite risk score, worst material group name, spend concentration percentage, and total PO spend.")


Q: Who are our top 3 highest-risk suppliers in 2020? For each, show their supplier name, composite risk score, worst material group name, spend concentration percentage, and total PO spend.
Operation  : get_supplier_risk
Parameters : {
  "year": "2020",
  "order_by": "CompositeRiskScore desc",
  "top": 3,
  "select": "Supplier,Year,CompositeRiskScore,WorstMaterialGroupName,SpendConcentrationPct"
}
BDC Result : 3 row(s) from SAP Business Data Cloud - AM_SUPPLIER_RISK_SCORE

--- Round 2 ---
Operation  : get_supplier_info
Parameters : {
  "supplier": "USSU-VSF82",
  "top": 1,
  "select": "Supplier,SupplierName"
}
BDC Result : 1 row(s) from SAP Business Data Cloud - V_SUPPLIER
Operation  : get_supplier_info
Parameters : {
  "supplier": "USSU-VSF85",
  "top": 1,
  "select": "Supplier,SupplierName"
}
BDC Result : 1 row(s) from SAP Business Data Cloud - V_SUPPLIER
Operation  : get_supplier_info
Parameters : {
  "supplier": "0001000412",
  "top": 1,
  "select": "Supplier,SupplierName"
}
BDC R

Here are the top 3 highest-risk suppliers in 2020 and the requested details:

| Rank | Supplier ID  | Supplier Name               | Composite Risk Score | Worst Material Group | Spend Concentration | 2020 PO Spend | Currency |
|------|---------------|-----------------------------|----------------------|----------------------|---------------------|---------------|----------|
| 1    | USSU-VSF82    | Supplier USSU-VSF82         | 45                   | Frame                | 100.0%              | 72,956        | USD      |
| 2    | USSU-VSF85    | Supplier ussu-vsf85         | 45                   | Material group 1     | 100.0%              | 11,500        | USD      |
| 3    | 0001000412    | ARGENTINA NETWORKS GENERADORA | 28                   | N/A                  | N/A                 | 0             | —        |

Notes:
- 2020 PO spend is the sum of NetAmount from PO history for 2020. Supplier 0001000412 has no 2020 PO history rows; spend shown as 0.
- Spend concentration is as returned by the risk score dataset; N/A indicates no percentage available.

Source: SAP Business Data Cloud (AM_SUPPLIER_RISK_SCORE, V_SUPPLIER, AM_PO_HISTORY). Retrieved 2026-09-29T20:56:43Z.

'Here are the top 3 highest-risk suppliers in 2020 and the requested details:\n\n| Rank | Supplier ID  | Supplier Name               | Composite Risk Score | Worst Material Group | Spend Concentration | 2020 PO Spend | Currency |\n|------|---------------|-----------------------------|----------------------|----------------------|---------------------|---------------|----------|\n| 1    | USSU-VSF82    | Supplier USSU-VSF82         | 45                   | Frame                | 100.0%              | 72,956        | USD      |\n| 2    | USSU-VSF85    | Supplier ussu-vsf85         | 45                   | Material group 1     | 100.0%              | 11,500        | USD      |\n| 3    | 0001000412    | ARGENTINA NETWORKS GENERADORA | 28                   | N/A                  | N/A                 | 0             | —        |\n\nNotes:\n- 2020 PO spend is the sum of NetAmount from PO history for 2020. Supplier 0001000412 has no 2020 PO history rows; spend shown as 0.\n- Spend concentrati

In [75]:
# Q10 (Complex) — Three: RISK + PO_HISTORY + V_SUPPLIER
ask("Compare the year-over-year risk trend for Supplier 0017300001: show its risk category,composite risk score, spend concentration, single-source count, total PO spend, and late delivery rate for 2018, 2019, and 2020. and what city, region supplier is located?") 


Q: Compare the year-over-year risk trend for Supplier 0017300001: show its risk category,composite risk score, spend concentration, single-source count, total PO spend, and late delivery rate for 2018, 2019, and 2020. and what city, region supplier is located?
Operation  : get_supplier_risk
Parameters : {
  "supplier": "0017300001",
  "years": "2018,2019,2020",
  "select": "Supplier,Year,RiskCategory,CompositeRiskScore,SpendConcentrationScore,SpendConcentrationPct,SingleSourcedMaterialCount",
  "top": 50
}
BDC Result : 3 row(s) from SAP Business Data Cloud - AM_SUPPLIER_RISK_SCORE
Operation  : get_po_history
Parameters : {
  "supplier": "0017300001",
  "year": "2018",
  "select": "NetAmount,Late_Delivery_Pct,CountScheduleLines,LocalCurrency",
  "top": 5000
}
BDC Result : 1 row(s) from SAP Business Data Cloud - AM_PO_HISTORY
Operation  : get_po_history
Parameters : {
  "supplier": "0017300001",
  "year": "2019",
  "select": "NetAmount,Late_Delivery_Pct,CountScheduleLines,LocalCurrency"

Here’s the year-over-year view for Supplier 0017300001.

| Year | Risk Category | Composite Risk Score | Spend Concentration (%) | Single-Source Count | Total PO Spend (USD) | Late Delivery Rate |
|------|---------------|----------------------|-------------------------|---------------------|----------------------|--------------------|
| 2018 | MEDIUM        | 46                   | 100.0%                  | 1                   | 77,135.20            | 0.0%               |
| 2019 | MEDIUM        | 46                   | 99.9%                   | 2                   | 51,900,490,941.70    | 0.0%               |
| 2020 | LOW           | 12                   | N/A                     | 0                   | N/A                  | N/A                |

Notes:
- 2020 PO history: no records found, so total spend and late delivery rate are not available.
- Spend concentration shown as percentage of group spend captured by this supplier (per risk dataset).

Supplier location:
- HighTec Assembling Corp., Muncie, IN, US

Sources and retrieval timestamps:
- SAP Business Data Cloud – AM_SUPPLIER_RISK_SCORE (Supplier risk): 2026-09-29T20:46:29Z
- SAP Business Data Cloud – AM_PO_HISTORY (PO history): 2018 query 2026-09-29T20:46:31Z; 2019 query 2026-09-29T20:46:34Z; 2020 query 2026-09-29T20:46:36Z
- SAP Business Data Cloud – V_SUPPLIER (Supplier master): 2026-09-29T20:46:37Z

'Here’s the year-over-year view for Supplier 0017300001.\n\n| Year | Risk Category | Composite Risk Score | Spend Concentration (%) | Single-Source Count | Total PO Spend (USD) | Late Delivery Rate |\n|------|---------------|----------------------|-------------------------|---------------------|----------------------|--------------------|\n| 2018 | MEDIUM        | 46                   | 100.0%                  | 1                   | 77,135.20            | 0.0%               |\n| 2019 | MEDIUM        | 46                   | 99.9%                   | 2                   | 51,900,490,941.70    | 0.0%               |\n| 2020 | LOW           | 12                   | N/A                     | 0                   | N/A                  | N/A                |\n\nNotes:\n- 2020 PO history: no records found, so total spend and late delivery rate are not available.\n- Spend concentration shown as percentage of group spend captured by this supplier (per risk dataset).\n\nSupplier location:\n- Hi